# First simple prediction model

This first model serves as a baseline. 
Many other machine-learning algorithms are compatible with the scikit-learn interface (e.g., [XGBoost](https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html)). 

Therefore, switching between models requires only minimal changes to the code.

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

## Data

### Load Data
Load race results from the last 10 seasons (2016 - 2025) and exclude non-starting drivers. 

Replace missing starting and qualifying positions with 25 *(debatable)*.

In [ ]:
df = pd.read_csv("data/raw/f1db-races-race-results.csv")

# select years and exclude DNS, DNQ, DNP and DNPQ
df = df[[2016 <= year <= 2025 and position not in ["DNS", "DNQ", "DNP", "DNPQ"] for year, position in zip(df["year"], df["positionText"])]].copy()

# replace missing starting and qualifying positions
df["gridPositionNumber"] = df["gridPositionNumber"].fillna(25)
df["qualificationPositionNumber"] = df["qualificationPositionNumber"].fillna(25)

C:\Users\floba\AppData\Local\Temp\ipykernel_2776\3757766761.py:1: DtypeWarning: Columns (31: fastestLap, 32: driverOfTheDay) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/raw/f1db-races-race-results.csv")


In [13]:
df.head(10)

,raceId,year,round,positionDisplayOrder,positionNumber,positionText,driverNumber,driverId,constructorId,engineManufacturerId,...,polePosition,qualificationPositionNumber,qualificationPositionText,gridPositionNumber,gridPositionText,positionsGained,pitStops,fastestLap,driverOfTheDay,grandSlam
22970,936,2016,1,1,1.0,1,6,nico-rosberg,mercedes,mercedes,...,False,2.0,2,2.0,2,1.0,2.0,False,False,False
22971,936,2016,1,2,2.0,2,44,lewis-hamilton,mercedes,mercedes,...,True,1.0,1,1.0,1,-1.0,2.0,False,False,False
22972,936,2016,1,3,3.0,3,5,sebastian-vettel,ferrari,ferrari,...,False,3.0,3,3.0,3,0.0,3.0,False,False,False
22973,936,2016,1,4,4.0,4,3,daniel-ricciardo,red-bull,tag-heuer,...,False,8.0,8,8.0,8,4.0,3.0,True,False,False
22974,936,2016,1,5,5.0,5,19,felipe-massa,williams,mercedes,...,False,6.0,6,6.0,6,1.0,2.0,False,False,False
22975,936,2016,1,6,6.0,6,8,romain-grosjean,haas,ferrari,...,False,19.0,19,19.0,19,13.0,1.0,False,True,False
22976,936,2016,1,7,7.0,7,27,nico-hulkenberg,force-india,mercedes,...,False,10.0,10,10.0,10,3.0,2.0,False,False,False
22977,936,2016,1,8,8.0,8,77,valtteri-bottas,williams,mercedes,...,False,11.0,11,16.0,16,8.0,2.0,False,False,False
22978,936,2016,1,9,9.0,9,55,carlos-sainz-jr,toro-rosso,ferrari,...,False,7.0,7,7.0,7,-2.0,3.0,False,False,False
22979,936,2016,1,10,10.0,10,33,max-verstappen,toro-rosso,ferrari,...,False,5.0,5,5.0,5,-5.0,3.0,False,False,False


### Variable selection 
This very simple model takes starting position, qualifying position, driver and team as predictor variables and result as a predicted variable. 

In [7]:
# predictor variables
# pd.get_dummies converts categorical variables (driverId and constructorId) into numerical indicators
X = pd.get_dummies(df[["gridPositionNumber", "qualificationPositionNumber", "driverId", "constructorId"]])

# predicted variable
y = df["positionDisplayOrder"]

In [8]:
X.head()

,gridPositionNumber,qualificationPositionNumber,driverId_alexander-albon,driverId_antonio-giovinazzi,driverId_brendon-hartley,driverId_carlos-sainz-jr,driverId_charles-leclerc,driverId_daniel-ricciardo,driverId_daniil-kvyat,driverId_esteban-gutierrez,...,constructorId_mclaren,constructorId_mercedes,constructorId_racing-bulls,constructorId_racing-point,constructorId_rb,constructorId_red-bull,constructorId_renault,constructorId_sauber,constructorId_toro-rosso,constructorId_williams
22970,2.0,2.0,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
22971,1.0,1.0,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
22972,3.0,3.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
22973,8.0,8.0,False,False,False,False,False,True,False,False,...,False,False,False,False,False,True,False,False,False,False
22974,6.0,6.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [9]:
y.head()

22970    1
22971    2
22972    3
22973    4
22974    5
Name: positionDisplayOrder, dtype: int64

### Training and Test Set
Define 2016 - 2024 as training data and 2025 as test data. 

Create a Boolean vector to identify training and test observations, which can then be used to select the corresponding rows.

In [ ]:
# create boolean vectors
train = df["year"] < 2025
test = df["year"] == 2025

## Model

### Training
Train the model using a random forest.

In [14]:
trees = 100
seed = 123

In [16]:
# setup model
model = RandomForestRegressor(n_estimators=trees, random_state=seed)

# training
model.fit(X[train], y[train]) # select training data rows using the train Boolean vector

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",123
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease 

### Prediction

In [17]:
# save the actual results using the test Boolean vector 
results = df.loc[test, ["raceId", "driverId", "positionDisplayOrder"]].copy()

# prediction returning a NumPy array containing a float for every driver (save them in the same result df)
results["prediction"] = model.predict(X[test]) # using test Boolean vector

The prediction outupts a NumPy array containing floats with the predicted rank for every driver (e.g. 2.4).

Rank the drivers within each race:

In [18]:
# grouping all predictions belonging to the same race
grouped = results.groupby("raceId")["prediction"]

# rank drivers within each race
ranks = grouped.rank(method="first") # whole numbers that get stored as floats

# convert floats (ranks) to integers and save them in the results df
results["predicted_position"] = ranks.astype(int)

### Evaluation
Compare the actual and the predicted ranks using the MAE (mean absolute error).

In [19]:
print(mean_absolute_error(results["positionDisplayOrder"], results["predicted_position"]))

3.7478991596638656


The MAE should be interpreted as: 

*In average, how many ranks are we away from the actual result?*

### Save Predictions

In [ ]:
# results.to_csv("predictions_2025.csv", index=False)